In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))
pd.set_option("display.max_columns", None)


from src.utils.basics_info import display_basic_info, extend_date, distance_bucket

In [2]:
path_rossmann = "../data/Rossmann Stores Data.csv"
path_store = "../data/store.csv"
df_rossmann = pd.read_csv(path_rossmann)
df_store = pd.read_csv(path_store)

/tmp/ipykernel_18264/432379514.py:3: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rossmann = pd.read_csv(path_rossmann)


# Understanding the variables

In [3]:
df_rossmann.head(3)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1


In [4]:
df_store.head(3)

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"


In [5]:
df_merged = df_rossmann.merge(df_store, how="left", on="Store")
df_merged.head(3)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"


In [6]:
display_basic_info(df_merged)

shape: (1017209, 18)
columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']
duplicates: 0

INFO:
<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Store                      1017209 non-null  int64  
 1   DayOfWeek                  1017209 non-null  int64  
 2   Date                       1017209 non-null  str    
 3   Sales                      1017209 non-null  int64  
 4   Customers                  1017209 non-null  int64  
 5   Open                       1017209 non-null  int64  
 6   Promo                      1017209 non-null  int64  
 7   StateHoliday               1017209 non-null  

# Initial Observations

```
- Dataset: 1M+ rows, 18 columns
- No duplicate records
- Missing values in competition & promo features
- Date column needs conversion
- Sales contain zero values (possible closed stores)
- Open feature indicates ~17% records where store is closed
- Competition and Promo2 features likely have conditional missing values (not random)
- Dataset contains both categorical and numerical features suitable for ML modeling
```

# Data Wrangling

In [7]:
print("Data distribution of open closed:")
print(f"{df_merged['Open'].value_counts()}")
df = df_merged[df_merged["Open"]==1]
df = df.drop("Open", axis=1)
df["StateHoliday"] = df["StateHoliday"].astype(str)
df = extend_date(df = df.copy(), colname="Date")
df.head(3)

Data distribution of open closed:
Open
1    844392
0    172817
Name: count, dtype: int64


,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,year,month,day,week_of_year
0,1,5,2015-07-31,5263,555,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2015,7,31,31
1,2,5,2015-07-31,6064,625,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2015,7,31,31
2,3,5,2015-07-31,8314,821,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",2015,7,31,31


In [8]:
top10storesWithHighestSales = df.groupby(['Store'])[['Sales', 'Customers']].sum().sort_values(ascending=False, by='Sales').head(10)
bottom10storesWithLeastSales = df.groupby(['Store'])[['Sales', 'Customers']].sum().sort_values(ascending=True, by='Sales').head(10)
meanSalesForStateHoliday = df.groupby(['StateHoliday'])['Sales'].mean().sort_values(ascending=False)
meanSalesForSchoolHoliday = df.groupby(['SchoolHoliday'])['Sales'].mean().sort_values(ascending=False)
meanSalesForDayOfWeek = df.groupby(['DayOfWeek'])['Sales'].mean().sort_values(ascending=False)
meanSalesForPromo = df.groupby(['Promo'])['Sales'].mean().sort_values(ascending=False)

In [9]:
print("Average sales :",df["Sales"].mean())

Average sales : 6955.514290755952


In [10]:
top10storesWithHighestSales

,Sales,Customers
Store,,
262,19516842,3204694
817,17057867,2454370
562,16927322,2924960
1114,16202585,2509542
251,14896870,1908934
513,14252406,1643527
788,14082141,1346835
733,14067158,3206058
383,13489879,1720249


In [11]:
bottom10storesWithLeastSales

,Sales,Customers
Store,,
307,2114322,240704
543,2179287,187583
198,2268273,264690
208,2302052,324162
263,2306075,221342
841,2318635,321495
879,2340576,216037
254,2341661,201507
186,2353548,237019


In [12]:
meanSalesForStateHoliday.reset_index(name='MeanSale')

,StateHoliday,MeanSale
0,b,9887.889655
1,c,9743.746479
2,a,8487.471182
3,0,6953.515034


In [13]:
meanSalesForSchoolHoliday.reset_index(name='MeanSale')

,SchoolHoliday,MeanSale
0,1,7200.181650
1,0,6896.782411


In [14]:
meanSalesForPromo.reset_index(name='MeanSale')

,Promo,MeanSale
0,1,8228.281239
1,0,5929.407603


In [15]:
meanSalesForDayOfWeek.reset_index(name='MeanSale')

,DayOfWeek,MeanSale
0,7,8224.723908
1,1,8216.073074
2,2,7088.113656
3,5,7072.677012
4,4,6767.310159
5,3,6728.122978
6,6,5874.840238


In [16]:
df = distance_bucket(df.copy())
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,year,month,day,week_of_year,CompetitionDistanceBucket
0,1,5,2015-07-31,5263,555,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2015,7,31,31,close
1,2,5,2015-07-31,6064,625,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2015,7,31,31,very_close
2,3,5,2015-07-31,8314,821,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",2015,7,31,31,far
3,4,5,2015-07-31,13995,1498,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN,2015,7,31,31,very_close
4,5,5,2015-07-31,4822,559,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN,2015,7,31,31,far


In [17]:
df.groupby(["CompetitionDistanceBucket"])["Sales"].mean()

CompetitionDistanceBucket
very_close    7301.654210
close         6836.022135
medium        6742.587896
far           6824.760993
Name: Sales, dtype: float64

In [18]:
df.to_csv("../data/merged.csv", index=False)

## 📊 Exploratory Analysis Insights

### 🏪 Store Status (Open/Closed)
```- ~83% stores are open, ~17% closed
- Closed stores likely contribute to zero sales
👉 Consider filtering `Open = 1` for model training
```

---

### 🏬 Top Performing Stores
```
- Few stores (e.g., Store 262, 817) contribute very high sales
- Significant variation across stores
👉 Store-level behavior is important (consider store-based features)
```

---

### 🎉 State Holiday Impact
```
- Highest sales during holidays: b, c
- Lowest sales on non-holiday (0)
👉 Holidays have strong impact on sales
```

---

### 🏫 School Holiday Impact
```
- Slight increase in sales during school holidays
👉 Moderate influence on customer activity
```

---

### 📢 Promotion Impact
```
- Sales with Promo = ~8228 vs without Promo = ~5929
👉 Promotions significantly boost sales (very important feature)
```

---

### 📅 Day of Week Trends
```
- Highest sales on Day 7 and Day 1
- Lowest sales on Day 6
👉 Strong weekly seasonality pattern
```
### Competition Distance
```
- Highest sales for very closed competition distance
- Sales decreasing when distance increasing
```
---

## ⚠️ Key Takeaways
```
- Promotions and holidays are major drivers of sales
- Store-level variation is significant
- Weekly patterns exist → time-based features important
```